# Experiment Folder Creator
This file is for loading in a base config from ./base_configs, modifying a few of the fields, and then creating the experiment folder

## Specify Base Config

In [17]:
## Load in base config
from pathlib import Path
import yaml
import copy
import os

BASE_CONFIG_PATH = Path("base_configs/farming_basic/fb_10_arms.yaml")
# Load yaml file in as dictionary
with open(BASE_CONFIG_PATH, "r") as f:
    base_config = yaml.safe_load(f)

# Quick sanity display (in a notebook this will print the dict)
# base_config


## Helper Update Functions

In [8]:
def deep_update(original, update):
    """Deep update original dict with values from the update dict."""
    for key, value in update.items():
        if isinstance(value, dict):
            original[key] = deep_update(original.get(key, {}), value)
        else:
            original[key] = value
    return original

def save_config(config, directory, experiment_name):
    full_path = os.path.join(directory, experiment_name)
    os.makedirs(full_path, exist_ok=True)
    with open(os.path.join(full_path, "config.yaml"), 'w') as f:
        yaml.dump(config, f)

In [9]:
def generate_experiment_name(params):
    # Convert each parameter to a string of the form "paramName_paramValue"
    # and join them all with underscores
    return "_".join([f"{param}_{value}" for param, value in params.items()])

In [5]:
# This config isn't run, but rather just points to the subexperiments - TODO: Decide if this is still relevant
def save_base_config(base_config, experiment_set):
    # Update the master_path and dir keys
    base_config['paths']['eval_results_master_path'] = f"experiments/{experiment_set}/eval_results.csv"
    base_config['paths']['experiment_dir'] = f"experiments/{experiment_set}"
    
    # Write the updated base config to the experiment set directory
    with open(f"experiments/{experiment_set}/config.yaml", 'w') as f:
        yaml.dump(base_config, f)



## Shuffle tools

In [22]:
# Here we add order shuffling for actions
import math, random, itertools

# Lehman unranking for huge spaces - from chatgpt, haven't checked validity
def unrank_permutation(elems, rank):
            elems = list(elems)
            n = len(elems)
            out = []
            r = rank
            for i in range(n-1, -1, -1):
                fact = math.factorial(i)
                idx = r // fact
                r %= fact
                out.append(elems.pop(idx))
            return tuple(out)

def sample_k_orderings(keys, k, *, seed=None):
    keys = list(keys)
    n = len(keys)
    total = math.factorial(n)
    if k > total:
        raise ValueError(f"k ({k}) > number of possible orderings ({total})")
    if seed is not None:
        random.seed(seed)
    # If total is small-ish, enumerate and sample (simple & reproducible)
    if total <= 1000000:
        perms = list(itertools.permutations(keys))
        return random.sample(perms, k)
    else:
        ranks = random.sample(range(total), k)
        return [unrank_permutation(keys, r) for r in ranks]

## Value mapping table

In [11]:
# Define the mapping from parameter values to specific config changes
value_mapping = {
        'time_horizon': {
            5: {'experiment': {'time_horizon': 5}},
            10: {'experiment': {'time_horizon': 10}},
            20: {'experiment': {'time_horizon': 20}}
        },
        'agent': {
            'MonoLLM': {'agent': {'type': 'mono_llm'}},
        },
        'model_name': {
            'Qwen2.5-7B-Instruct': {'agent': {'model_name': "Qwen/Qwen2.5-7B-Instruct"}},
            'Qwen2.5-14B-Instruct': {'agent': {'model_name': "Qwen/Qwen2.5-14B-Instruct"}},
            'Qwen2.5-32B-Instruct': {'agent': {'model_name': "Qwen/Qwen2.5-32B-Instruct"}},
            'Qwen2.5-72B-Instruct': {'agent': {'model_name': "Qwen/Qwen2.5-72B-Instruct"}},
        },
        
}

## Param grid - THIS IS WHERE YOU UPDATE
Specify the variations which you want in your experiments. If you specify a single value for a variable (e.g. 'noise': [0]), then all generated configs will have that value. If you specify multiple values (e.g. 'noise': [0, 0.25]), then configs will be generated with each of those values. For example, if you specify 'noise': [0, 0.25] and 'selection': ['ucb', 'thompson'], then you will generate 4 config files - one with each pairwise combination.


In [18]:
# Define the parameter grid
param_grid = {
        # 'llm_temp': [1],
        'model_name': ['Qwen2.5-32B-Instruct'],
        'agent': ['MonoLLM'],
        'time_horizon': [10],
    }


In [19]:
experiment_set = f'20251104_fb_10arms'

In [20]:
# How many times to shuffle the order of the actions (without replacement)
order_shuffles_max = 6

## This function will create the folder

Make sure everything is in order before running. Also make sure your experiment folder name is correctly set to avoid overwriting another folder.


In [23]:
from itertools import permutations, product
import math

# Load the base config file
with open(BASE_CONFIG_PATH, 'r') as f:
    base_config = yaml.safe_load(f)

# Generate and save config files for each combination
for idx, param_values in enumerate(product(*param_grid.values())):
        param_values_dict = dict(zip(param_grid.keys(), param_values))
        #print(param_values_dict)
        experiment_name = generate_experiment_name(param_values_dict)
        updated_config = yaml.safe_load(yaml.dump(base_config))  # deep copy

        # Apply updates to the config based on the current parameter values
        for param, value in param_values_dict.items():
            if param in value_mapping and value in value_mapping[param]:
                deep_update(updated_config, value_mapping[param][value])
            else:
                print(f"No mapping found for parameter '{param}' with value '{value}'")

        # Here we add order shuffling for actions
        # Two options - one is itertools.permutations which gives all possible permutations
        # The other is randomly drawing permutations, and checcking for collisions. I think itertools
        # is better when num_actions is small, so we'll do that for now

        if order_shuffles_max == 0: # Don't save the config
            # Add logging file
            updated_config['paths']['log_file'] = os.path.join(f'experiments/{experiment_set}', experiment_name, "output.log")
            save_config(updated_config, f'experiments/{experiment_set}', experiment_name)
        else:
            actions = updated_config.get('actions')
            if actions and isinstance(actions, list) and len(actions) > 1:
                k = min(order_shuffles_max, math.factorial(len(actions)))
                orderings = sample_k_orderings(actions, k, seed=42)
                for ord_idx, ordering in enumerate(orderings):
                    cfg_copy = yaml.safe_load(yaml.dump(updated_config))  # deep copy
                    cfg_copy['actions'] = list(ordering)
                    name_with_order = f"{experiment_name}_order{ord_idx}"
                    cfg_copy['paths']['log_file'] = os.path.join(f"experiments/{experiment_set}/{experiment_name}", name_with_order, "output.log")
                    save_config(cfg_copy, f"experiments/{experiment_set}/{experiment_name}", name_with_order)

        

# Create a copy of param_grid with only the single item lists and do a deep update
new_base_config = yaml.safe_load(yaml.dump(base_config))  # deep copy
for param_key, param_value in param_grid.items():
    if len(param_value) == 1: # If there is only a single value for this param, change the base config to have it
        if param in value_mapping and value in value_mapping[param]:
            deep_update(new_base_config, value_mapping[param_key][param_value[0]])
         

#saves base config for evaluation purposes        
save_base_config(new_base_config, experiment_set)



In [ ]:
len(list(product(*param_grid.values())))